In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)

In [ ]:
# 1) 데이터 로드
df1 = pd.read_csv('../../data_processed/product_type_1.csv',header=[0,1])
df2 = pd.read_csv('../../data_processed/product_type_2.csv',header=[0,1])

In [ ]:
def is_multiindex_columns(df: pd.DataFrame) -> bool:
    return isinstance(df.columns, pd.MultiIndex)

def get_block(df: pd.DataFrame, block_name: str) -> pd.DataFrame:
    if is_multiindex_columns(df):
        return df[block_name].copy()
    prefix = block_name.lower() + "_"
    cols = [c for c in df.columns if str(c).lower().startswith(prefix)]
    return df[cols].copy()

process_df = get_block(df1, "process")
sensor_df  = get_block(df1, "sensor")
defects_df = get_block(df1, "defects")

print("process_df:", process_df.shape)
print("sensor_df :", sensor_df.shape)
print("defects_df:", defects_df.shape)
print("\n[defects columns]")
print(defects_df.columns.tolist())


In [ ]:
defects_numeric = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_bin = (defects_numeric > 0).astype(int)

defect_groups = {
    "표면": ["dent_1", "stain_1", "exfoliation_1", "exfoliation_2"],
    "구조": ["short_shot_1", "short_shot_2", "bubble_1", "bubble_2", "deformation_1", "deformation_2"],
}

def normalize_defect_colname(col):
    s = str(col).lower()
    return s.replace("defects_", "") if s.startswith("defects_") else s

map_to_real = {normalize_defect_colname(c): c for c in defects_bin.columns}

y = pd.DataFrame(index=df2.index)
for group_name, cols in defect_groups.items():
    real_cols = [map_to_real[c] for c in cols if c in map_to_real]
    if len(real_cols) == 0:
        raise KeyError(f"[{group_name}] 그룹에 해당하는 defects 컬럼을 찾지 못했습니다: {cols}")
    y[group_name] = defects_bin[real_cols].any(axis=1).astype(int)

print("[표면/구조 불량 비율(%)]")
display((y.mean() * 100).round(2).to_frame("rate_%"))
y.head()


In [ ]:
X = pd.concat([process_df, sensor_df], axis=1).copy()
X = X.drop(columns=["air_pressure_min", "air_pressure_max", "coolant_temp_min", "coolant_temp_max", 
                    "factory_temp_min", "factory_temp_max", "factory_humidity_min", "factory_humidity_max"], errors="ignore")
print("X shape (before drop):", X.shape)

def col_to_str(c):
    if isinstance(c, tuple):
        return "_".join([str(x) for x in c]).lower()
    return str(c).lower()

leak_keywords = [
    "process_id",
    "process_shot",
    "process_product_type",
    "defect_flag_is_defect",
    "is_defect",
]

leak_cols = [c for c in X.columns if any(k in col_to_str(c) for k in leak_keywords)]
if leak_cols:
    print("🚫 drop leakage cols:", leak_cols)
    X = X.drop(columns=leak_cols, errors="ignore")
else:
    print("✅ leakage 키워드 컬럼 없음")

defect_like = [c for c in X.columns if "defects" in col_to_str(c)]
if defect_like:
    print("🚫 drop defects-like cols from X:", defect_like)
    X = X.drop(columns=defect_like, errors="ignore")

print("X shape (after drop):", X.shape)


In [ ]:
strata = y.astype(str).agg("".join, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

print("\n[라벨 조합 분포(전체)]")
display(strata.value_counts(normalize=True).sort_index().to_frame("ratio"))


In [ ]:
X_train.isna().sum()

In [ ]:
df2 = pd.read_csv("../../data_processed/train_2.csv")

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=df2.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')

강한 양의 상관관계
- velocity_1, velocity_2 (0.81), velocity_2, velocity_3 (0.71) velocity_1, velocity_3 (0.65)
    - 속도 관련 지표들이 서로 밀접하게 연동되어 있음
    - 1과 2 사이는 0.81로 매우 강한 상관관계를 보이지만, 1과 3 사이는 0.65로 낮아진다.
        - 공정 단계별로 속도가 순차적 측정됨
        - 인접한 단계끼리 더 밀접하게 연동됨
            - 다중공선성 문제 발생 가능?
- cylinder_pressure, casting_pressure (1.00)
    - 실린더 압력이 가해질 때 주조 압력이 정비례하며 상승하는 공정 특징
    - 실린더 압력이 곧 주조 압력으로 이어지는 기계적 구조
        - 하나는 삭제해도 무관??
- cylinder_pressure, spray_time (0.74)
    - 특정 압력 조건에 도달하기 위해 스프레이 분사 시간 조절하거나, 압력이 높을 때 공정 안정화를 위해 스프레이 시간을 길게 가져가는 제어 로직이 존재함을 의미
- shot, coolant_temp (0.71)
    - shot 횟수가 늘어날수록 기계 내부의 마찰과 사출 과정에서 발생하는 열로 인해 냉각수 온도가 자연스럽게 상승?
        - 열 축적 현상?
            - shot 수치가 늘어날수록 냉각 효율이 어떻게 변하는가?
            - 특정 shot 구간에서 온도가 임계치를 넘는가?
- casting_pressure, spray_time (0.74), casting_pressure, spray_1_time (0.62)  
    - 전체 분사 시간(spray_time)이 개별 단계인 spray_1_time 보다 압력과 더 밀접
        - 압력 형성이 특정 단계보다는 전체적인 분사 공정 환경에 더 크게 의존함
- spray_time, spray_1_time (0.63)
    - spray_time 내에서 spray_1_time이 차지하는 비중이 크거나 spray_1_time이 길어지면 전체 공정 시간도 함께 늘어난다는 인과관계 반영
        - 1.00이 아니므로 spray_2_time 등 다른 단계의 변동성도 존재함을 시사



강한 음의 상관계수

- cycle_time, high_velocity(-0.67) 
    - high_velocity가 빠를수록 제품이 나오기까지 걸리는 총 시간이 줄어드는 관계
    - 금형에 용탕을 투입하는 속도를 높임으로써 생산성을 높이는 직접적 인과관계
- spray_2_time, cylinder_pressure(-0.71), spray_2_time, casting_pressure(-0.71)
    - 스프레이 작업 시간(spray_2_time)이 길어질수록 Shot을 찍기 위한 실린더 압력 및 Shot을 찍을 때의 압력이 낮아지는 경향
    - 스프레이 작업 시간이 길어지면 액체의 온도가 낮아져 내부 압력이 떨어지거나 압력이 충분히 형성되지 않았을 때 보완을 위해 스프레이를 더 오래 수행하는 제어 패턴
- coolant_pressure, cylinder_pressure(-0.68), coolant_pressure, casting_pressure(-0.68)
    - 냉각수 압력 측정값이 높을수록 Shot을 찍기 위한 실린더 압력 및 Shot을 찍을 때의 압력이 낮아지는 경향
- coolant_pressure, spray_time(-0.65), coolant_pressure, spray_1_time(-0.71)
    - 냉각수 압력 측정값이 높을수록 스프레이 작업 시간(spray_time, spray_1_time)이 줄어드는 경향
- spray_2_time, spray_time(-0.59),
    - 스프레이 작업 시간이 서로 음??
- factory_humidity, factory_temp(-0.95)
    - 공장 온도 측정값이 높아지면 공장 습도 측정값이 낮아지는 경향
    - 전형적인 밀폐된 공장 내부의 열역학적 특성??

In [ ]:
# 1. 시각화를 위해 X_train과 y_train을 하나의 데이터프레임으로 합칩니다.
# y_train의 컬럼명을 'target'이라고 가정합니다.
train_df = pd.concat([X_train, y_train], axis=1)

# 2. 피어슨 상관계수 행렬 계산
corr_matrix = train_df.corr()

# 3. 히트맵 시각화
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap: X_train & y_train')
plt.show()

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=df1.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')